# 🛡️ HallucinationGuard — Proof of Concept on Qwen3.6-27B

**Goal**: prove the OpenInterp HallucinationGuard product works on Qwen3.6-27B before committing 90 days of product engineering. One Colab session, public benchmarks, falsifiable thresholds.

## The 3 questions we answer

| Question | Metric | Pass threshold |
|---|---|---|
| **Detection works?** | Probe AUROC on hallucination benchmarks | ≥ 0.70 (we already have 0.81 on Ferrando entity test) |
| **Mitigation works?** | Confident-wrong rate drop in `abstain` mode | ≥ 30% reduction, MMLU loss ≤ 3pp |
| **Competitive?** | Latency per scored token | < 50 ms |

## Foundation already shipped

- Paper-grade SAE: `caiovicentino1/qwen36-27b-sae-papergrade` (200M tokens, VE 0.71 at L31)
- Best hallucination feature: **L31 / f34957** — AUROC 0.8141 on Ferrando-style entity test (notebook 28)
- Baselines published: Linear probe ceiling L32=0.887, diff-of-means L32=0.859

## Outcome possible

🟢 **Pass all 3** → green light 90-day product roadmap. Notebook + numbers become the public marketing artifact.  
🟡 **Detection passes, mitigation partial** → tunable. MVP ships with caveats.  
🔴 **Detection fails on public benchmarks** → product is narrower than expected (entity-only). Pivot.

## Hardware

RTX 6000 Blackwell 96 GB on Colab Pro+. Total compute time: 1-2 h. Cost: ~R$10 in credits.

## What we ship from this notebook

- `HallucinationGuard` class (~150 lines, clean API) — lifts directly into `openinterp` PyPI package
- Headline plot for landing page
- JSON results pushed to HF Hub for reproducibility

In [ ]:
!pip -q install --upgrade transformers accelerate safetensors huggingface_hub datasets scipy scikit-learn matplotlib tqdm

## 1. Setup + auth

In [ ]:
import os, json, time, math, gc
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from huggingface_hub import login, hf_hub_download, HfApi, create_repo

HF_TOKEN = os.environ.get('HF_TOKEN')
if HF_TOKEN is None:
    import getpass
    HF_TOKEN = getpass.getpass('HF token (write scope): ')
login(HF_TOKEN, add_to_git_credential=False)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
assert device == 'cuda', 'Need GPU.'
print(f'CUDA: {torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# Config
CFG = {
    'model':            'Qwen/Qwen3.6-27B',
    'sae_repo':         'caiovicentino1/qwen36-27b-sae-papergrade',
    'sae_layer':        31,
    'hallucination_feature': 34957,    # best feature per notebook 28
    'subset_size': {
        'truthfulqa': 200,
        'haluval':    200,
        'simpleqa':   100,
        'mmlu':       500,
    },
    'guard_threshold':  0.7,            # initial — tune via sweep
    'max_new_tokens':   128,
    'gen_temperature':  0.0,            # greedy for reproducibility
    'hf_results_repo':  os.environ.get('HF_USERNAME', 'caiovicentino1') + '/hallucinationguard-proof-qwen36-27b',
}
LOCAL_OUT = Path('/content/hg_proof_out')
LOCAL_OUT.mkdir(parents=True, exist_ok=True)
print(json.dumps(CFG, indent=2, default=str))

## 2. Load Qwen3.6-27B

bf16 + SDPA. Frozen. Per memory `feedback_qwen35_training_stack.md`, Qwen3.5+ is multimodal — use `AutoModelForImageTextToText`. Memory budget: ~54 GB model + ~3 GB SAE + ~5 GB generation cache = comfortable on 96 GB.

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText, AutoModelForCausalLM

print(f'Loading {CFG["model"]} ...')
tok = AutoTokenizer.from_pretrained(CFG['model'], trust_remote_code=True)
try:
    model = AutoModelForImageTextToText.from_pretrained(
        CFG['model'],
        dtype=torch.bfloat16,
        attn_implementation='sdpa',
        device_map={'': device},
        trust_remote_code=True,
    )
except Exception:
    # fallback for pure-text variants
    model = AutoModelForCausalLM.from_pretrained(
        CFG['model'],
        dtype=torch.bfloat16,
        attn_implementation='sdpa',
        device_map={'': device},
        trust_remote_code=True,
    )
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

# Locate transformer block list
def _block_list(m):
    candidates = [m]
    if hasattr(m, 'model'):
        candidates.append(m.model)
    for s in candidates:
        for path in [('model','language_model','layers'), ('language_model','layers'),
                     ('model','layers'), ('layers',)]:
            cur = s; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                return cur
    raise RuntimeError('Could not find layer list.')

blocks = _block_list(model)
d_model = (model.config.text_config.hidden_size if hasattr(model.config, 'text_config')
           else model.config.hidden_size)
print(f'Model: {len(blocks)} layers, d_model = {d_model}')
print(f'GPU mem after load: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## 3. Load SAE L31

Pull from our paper-grade HF repo. Format: TopK SAE (n=65536, k=128, SAELens-compatible).

In [ ]:
from safetensors.torch import load_file

sae_path = hf_hub_download(
    repo_id=CFG['sae_repo'],
    filename=f"sae_L{CFG['sae_layer']}_latest.safetensors",
    token=HF_TOKEN,
)
cfg_path = hf_hub_download(
    repo_id=CFG['sae_repo'],
    filename=f"sae_L{CFG['sae_layer']}_cfg.json",
    token=HF_TOKEN,
)
sae_weights = load_file(sae_path)
sae_cfg = json.load(open(cfg_path))
print(f'SAE config: {sae_cfg}')
for k, v in sae_weights.items():
    print(f'  {k}: {tuple(v.shape)} {v.dtype}')

# Move to device, fp32 for numerical stability of single-feature score
W_enc = sae_weights['W_enc'].to(device).float()        # (d_model, n_features)
W_dec = sae_weights['W_dec'].to(device).float()        # (n_features, d_model)
b_enc = sae_weights['b_enc'].to(device).float()        # (n_features,)
b_dec = sae_weights['b_dec'].to(device).float()        # (d_model,)
n_features = W_enc.shape[1]
k_topk = sae_cfg.get('k', 128)
print(f'SAE: n={n_features}, k={k_topk}, target feature = f{CFG["hallucination_feature"]}')
print(f'GPU mem after SAE load: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## 4. The HallucinationGuard class

Clean, self-contained API that lifts directly into the `openinterp` PyPI package. Three modes:

- **`detect`**: score the prompt, return as-is. Score reported alongside output.
- **`warn`**: same as detect, but adds a flag in the output dict if score > threshold.
- **`abstain`**: if score > threshold, replace generation with a calibrated uncertainty response.

Score = **inverted** SAE feature `f34957` activation at the **last token of the prompt**.

`f34957` is an **"I-know-this-entity" feature**: it fires HIGH when the model recognizes an entity, and stays at 0 (TopK gating + ReLU) for unknown entities. We invert it so that **high HallucinationGuard score = low familiarity = high hallucination risk** — the natural product convention.

This was discovered empirically during the smoke test: known entities (Einstein, Tokyo) gave score 0.9-1.7, unknown entities gave score 0.0. Per notebook 28, this feature has AUROC 0.81 for known vs unknown discrimination — confirmed direction is "high = familiar".

In [ ]:
class HallucinationGuard:
    """Activation-probe-based hallucination detector + mitigator.
    
    Wraps any HF transformer with a hook at the SAE-supervised layer.
    Score = single SAE feature activation at last prompt token.
    
    Modes: 'detect' | 'warn' | 'abstain'
    """

    ABSTAIN_RESPONSE = (
        "I'm not confident about this. Please verify with an authoritative source."
    )

    def __init__(self, model, tokenizer, blocks, layer, W_enc, W_dec, b_enc, b_dec,
                 feature_idx, invert_score=True, device='cuda'):
        self.model = model
        self.tok = tokenizer
        self.blocks = blocks
        self.layer = layer
        self.W_enc = W_enc
        self.W_dec = W_dec
        self.b_enc = b_enc
        self.b_dec = b_dec
        self.feat_idx = feature_idx
        self.invert_score = invert_score   # f34957 fires on KNOWN entities, so we invert
        self.device = device
        self._buf = None
        self._hook = blocks[layer].register_forward_hook(self._capture_hook)

    def _capture_hook(self, _mod, _inp, out):
        h = out[0] if isinstance(out, tuple) else out
        self._buf = h.detach()

    def __del__(self):
        if hasattr(self, '_hook') and self._hook is not None:
            self._hook.remove()

    @torch.no_grad()
    def _residual_at_last_token(self, ids, attn_mask):
        """Forward through model, return residual at L31 for last valid token of each sample."""
        self._buf = None
        self.model(ids, attention_mask=attn_mask)
        h = self._buf                                                # (B, T, D) bf16
        last_pos = attn_mask.sum(dim=1) - 1                          # (B,) — last non-pad index
        last_h = h[torch.arange(h.size(0)), last_pos].float()        # (B, D)
        return last_h

    @torch.no_grad()
    def _score_residual(self, h_last):
        """SAE encode → return hallucination score. h_last: (B, D).
        
        If invert_score=True (default for f34957 "I-know-this" feature):
            high familiarity → low halluc score
            zero familiarity (unknown entity) → halluc score = 0
            Output range: (-inf, 0]. Higher = more likely to hallucinate.
        """
        pre = (h_last - self.b_dec[None]) @ self.W_enc + self.b_enc[None]   # (B, n_feat)
        acts = F.relu(pre)
        f_val = acts[:, self.feat_idx]                              # (B,)
        return -f_val if self.invert_score else f_val

    @torch.no_grad()
    def score(self, prompt, max_input_length=512):
        """Score a single prompt. Returns scalar float (raw feature activation)."""
        if isinstance(prompt, str):
            prompts = [prompt]
        else:
            prompts = list(prompt)
        enc = self.tok(prompts, return_tensors='pt', padding=True,
                       truncation=True, max_length=max_input_length).to(self.device)
        h_last = self._residual_at_last_token(enc['input_ids'], enc['attention_mask'])
        f_vals = self._score_residual(h_last)
        out = f_vals.cpu().tolist()
        return out[0] if isinstance(prompt, str) else out

    @torch.no_grad()
    def generate(self, prompt, mode='detect', threshold=0.7, max_new_tokens=128,
                 max_input_length=512, do_sample=False, temperature=1.0):
        """Generate with optional guard intervention. Returns dict."""
        assert mode in {'detect', 'warn', 'abstain'}
        # 1) score the prompt
        score = self.score(prompt, max_input_length=max_input_length)
        flagged = score > threshold
        # 2) abstain mode short-circuits if flagged
        if mode == 'abstain' and flagged:
            return {
                'text': self.ABSTAIN_RESPONSE,
                'score': score,
                'flagged': True,
                'mode': mode,
                'abstained': True,
            }
        # 3) generate normally
        enc = self.tok(prompt, return_tensors='pt', truncation=True,
                       max_length=max_input_length).to(self.device)
        gen_ids = self.model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else 1.0,
            pad_token_id=self.tok.pad_token_id or self.tok.eos_token_id,
        )
        new_text = self.tok.decode(gen_ids[0, enc['input_ids'].shape[1]:],
                                    skip_special_tokens=True)
        return {
            'text': new_text,
            'score': score,
            'flagged': bool(flagged) and mode in {'warn', 'abstain'},
            'mode': mode,
            'abstained': False,
        }

    def close(self):
        if self._hook is not None:
            self._hook.remove()
            self._hook = None

guard = HallucinationGuard(
    model=model, tokenizer=tok, blocks=blocks, layer=CFG['sae_layer'],
    W_enc=W_enc, W_dec=W_dec, b_enc=b_enc, b_dec=b_dec,
    feature_idx=CFG['hallucination_feature'],
    invert_score=True,    # f34957 is a familiarity feature; invert so high score = halluc risk
    device=device,
)
print(f'HallucinationGuard ready. invert_score={guard.invert_score} (high score → likely hallucination)')

## 5. Smoke test — known-known vs known-unknown entities

Before running benchmarks, verify the guard discriminates familiar from obscure entities. Per Ferrando methodology, asking about `'LeBron James'` should give low score (model knows him), and a fictional/obscure name should give high score.

In [ ]:
smoke_prompts = [
    ('known',   "What can you tell me about 'Albert Einstein'?"),
    ('known',   "What can you tell me about 'Barack Obama'?"),
    ('known',   "What can you tell me about 'Tokyo'?"),
    ('unknown', "What can you tell me about 'Bambale Osby'?"),       # real but obscure (basketball)
    ('unknown', "What can you tell me about 'Vlasik Korpel'?"),       # synthetic name
    ('unknown', "What can you tell me about 'Zephir Quartzhaven'?"),  # synthetic name
]

for label, prompt in smoke_prompts:
    s = guard.score(prompt)
    print(f'  [{label:7s}] halluc_score = {s:+7.3f}    {prompt}')

# With invert_score=True: known entities → score < 0 (familiar);  unknown → score ≈ 0 (no familiarity).
# Separation = unknown - known should be POSITIVE if guard works.
known_scores = [guard.score(p) for l,p in smoke_prompts if l == 'known']
unknown_scores = [guard.score(p) for l,p in smoke_prompts if l == 'unknown']
print(f'\nMean(known, model recognizes)    = {np.mean(known_scores):+.3f}    (should be NEGATIVE — low halluc risk)')
print(f'Mean(unknown, model hallucinates) = {np.mean(unknown_scores):+.3f}    (should be ~0 — high halluc risk)')
print(f'Separation = unknown - known       = {np.mean(unknown_scores) - np.mean(known_scores):+.3f}    '
      f'(positive = guard works as expected)')

## 6. Load public hallucination benchmarks

- **TruthfulQA-MC1** (200 questions): standard hallucination eval, multiple-choice
- **HaluEval-QA** (200): labelled correct vs hallucinated answers
- **SimpleQA** (100): entity-based factual QA — **best fit for our SAE feature**
- **MMLU** (500): capability control — must NOT regress

In [ ]:
from datasets import load_dataset
import random
random.seed(42)

def _safe_load(loader_fn, name):
    try:
        return loader_fn()
    except Exception as e:
        print(f'  [{name}] failed to load: {e}')
        return None

# TruthfulQA-MC1
tqa = _safe_load(
    lambda: load_dataset('truthful_qa', 'multiple_choice', split='validation'),
    'truthfulqa',
)
if tqa is not None:
    n = min(CFG['subset_size']['truthfulqa'], len(tqa))
    tqa_idx = random.sample(range(len(tqa)), n)
    tqa_subset = [tqa[i] for i in tqa_idx]
    print(f'  TruthfulQA-MC1: {len(tqa_subset)} questions')

# HaluEval QA
halu = _safe_load(
    lambda: load_dataset('pminervini/HaluEval', 'qa', split='data'),
    'haluval',
)
if halu is None:
    halu = _safe_load(
        lambda: load_dataset('Sneha/HaluEval', 'qa', split='train'),
        'haluval-fallback',
    )
if halu is not None:
    n = min(CFG['subset_size']['haluval'], len(halu))
    halu_idx = random.sample(range(len(halu)), n)
    halu_subset = [halu[i] for i in halu_idx]
    print(f'  HaluEval-QA: {len(halu_subset)} questions')

# SimpleQA — try OpenAI's repo first, fallback to community mirror
simpleqa = _safe_load(
    lambda: load_dataset('basicv8vc/SimpleQA', split='test'),
    'simpleqa',
)
if simpleqa is not None:
    n = min(CFG['subset_size']['simpleqa'], len(simpleqa))
    sq_idx = random.sample(range(len(simpleqa)), n)
    simpleqa_subset = [simpleqa[i] for i in sq_idx]
    print(f'  SimpleQA: {len(simpleqa_subset)} questions')

# MMLU (control — capability check)
mmlu = _safe_load(
    lambda: load_dataset('cais/mmlu', 'all', split='test'),
    'mmlu',
)
if mmlu is not None:
    n = min(CFG['subset_size']['mmlu'], len(mmlu))
    mmlu_idx = random.sample(range(len(mmlu)), n)
    mmlu_subset = [mmlu[i] for i in mmlu_idx]
    print(f'  MMLU: {len(mmlu_subset)} questions')

## 7. Baseline run — generate + grade

For each benchmark, run the model in baseline mode (no guard intervention), grade correctness, score the prompt with the guard. This gives us paired (correct, score) data per question for ROC analysis.

Grading rules:
- TruthfulQA-MC1: model picks index of most-truthful answer; check against label
- HaluEval-QA: dataset has `right_answer` and `hallucinated_answer`; we generate then check exact substring match against `right_answer`
- SimpleQA: short factual answer; lowercased substring match
- MMLU: model picks A/B/C/D; check against label

In [ ]:
from tqdm.auto import tqdm

def normalize_answer(s):
    return ''.join(ch.lower() for ch in str(s) if ch.isalnum() or ch.isspace()).strip()

@torch.no_grad()
def model_generate(prompt, max_new_tokens=128):
    enc = tok(prompt, return_tensors='pt', truncation=True, max_length=512).to(device)
    out = model.generate(
        **enc, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tok.pad_token_id or tok.eos_token_id,
    )
    return tok.decode(out[0, enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()

@torch.no_grad()
def model_pick_letter(prompt, choices, max_new_tokens=4):
    """For multi-choice: prepend choices, ask for letter, return predicted letter index."""
    letters = ['A', 'B', 'C', 'D'][:len(choices)]
    formatted = '\n'.join(f'{l}. {c}' for l, c in zip(letters, choices))
    p = f'{prompt}\n\n{formatted}\n\nAnswer (letter only):'
    out = model_generate(p, max_new_tokens=max_new_tokens)
    out = out.upper().strip()
    for l in letters:
        if out.startswith(l):
            return letters.index(l), p
    return -1, p   # could not parse

results = {'truthfulqa': [], 'haluval': [], 'simpleqa': [], 'mmlu': []}

# --- TruthfulQA-MC1 ---
if tqa is not None:
    for q in tqdm(tqa_subset, desc='truthfulqa baseline'):
        choices = q['mc1_targets']['choices']
        label = int(np.argmax(q['mc1_targets']['labels']))
        question = q['question']
        pred_idx, full_prompt = model_pick_letter(question, choices)
        score = guard.score(full_prompt)
        results['truthfulqa'].append({
            'question': question, 'pred': pred_idx, 'label': label,
            'correct': pred_idx == label, 'score': score,
        })
    correct = sum(r['correct'] for r in results['truthfulqa'])
    print(f'  TruthfulQA accuracy: {correct}/{len(results["truthfulqa"])} = {100*correct/len(results["truthfulqa"]):.1f}%')

# --- HaluEval-QA ---
if halu is not None:
    for q in tqdm(halu_subset, desc='haluval baseline'):
        question = q.get('question') or q.get('input')
        right = q.get('right_answer') or q.get('answer') or ''
        prompt = f'Q: {question}\nA:'
        gen = model_generate(prompt, max_new_tokens=64)
        score = guard.score(prompt)
        # exact substring match (lowercased, normalized)
        correct = normalize_answer(right) in normalize_answer(gen)
        results['haluval'].append({
            'question': question, 'gen': gen, 'right': right,
            'correct': bool(correct), 'score': score,
        })
    correct = sum(r['correct'] for r in results['haluval'])
    print(f'  HaluEval accuracy: {correct}/{len(results["haluval"])} = {100*correct/len(results["haluval"]):.1f}%')

# --- SimpleQA ---
if simpleqa is not None:
    for q in tqdm(simpleqa_subset, desc='simpleqa baseline'):
        question = q.get('problem') or q.get('question')
        right = q.get('answer') or q.get('correct')
        prompt = f'Q: {question}\nA:'
        gen = model_generate(prompt, max_new_tokens=64)
        score = guard.score(prompt)
        correct = normalize_answer(right) in normalize_answer(gen)
        results['simpleqa'].append({
            'question': question, 'gen': gen, 'right': right,
            'correct': bool(correct), 'score': score,
        })
    correct = sum(r['correct'] for r in results['simpleqa'])
    print(f'  SimpleQA accuracy: {correct}/{len(results["simpleqa"])} = {100*correct/len(results["simpleqa"]):.1f}%')

# --- MMLU ---
if mmlu is not None:
    for q in tqdm(mmlu_subset, desc='mmlu baseline'):
        choices = q['choices']
        label = q['answer']
        question = q['question']
        pred_idx, full_prompt = model_pick_letter(question, choices)
        score = guard.score(full_prompt)
        results['mmlu'].append({
            'question': question, 'pred': pred_idx, 'label': label,
            'correct': pred_idx == label, 'score': score,
        })
    correct = sum(r['correct'] for r in results['mmlu'])
    print(f'  MMLU accuracy: {correct}/{len(results["mmlu"])} = {100*correct/len(results["mmlu"]):.1f}%')

## 8. Detection AUROC — does the probe predict correctness?

For each benchmark, compute AUROC of `score → P(incorrect)`. AUROC > 0.5 = signal exists. AUROC > 0.7 = pass our threshold.

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

auroc_results = {}
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

for ax, (bench_name, rows) in zip(axes, results.items()):
    if not rows:
        ax.set_visible(False)
        continue
    # label = 1 if HALLUCINATED (incorrect)
    y_true = np.array([0 if r['correct'] else 1 for r in rows])
    y_score = np.array([r['score'] for r in rows])
    if len(np.unique(y_true)) < 2:
        ax.set_title(f'{bench_name}\n(degenerate — all {y_true[0]})')
        continue
    auroc = roc_auc_score(y_true, y_score)
    fpr, tpr, _ = roc_curve(y_true, y_score)
    auroc_results[bench_name] = float(auroc)
    ax.plot(fpr, tpr, color='#1f77b4', linewidth=2)
    ax.plot([0, 1], [0, 1], '--', color='gray', alpha=0.5)
    ax.set_title(f'{bench_name}  (AUROC = {auroc:.3f})')
    ax.set_xlabel('False positive rate')
    ax.set_ylabel('True positive rate')
    ax.grid(alpha=0.3)

plt.suptitle('HallucinationGuard detection AUROC — per benchmark', fontsize=14)
plt.tight_layout()
fig.savefig(LOCAL_OUT / 'detection_auroc.png', dpi=200, bbox_inches='tight')
plt.show()

print('\nQuestion 1: Detection works?')
for name, auroc in auroc_results.items():
    verdict = '✅' if auroc >= 0.70 else ('🟡' if auroc >= 0.6 else '❌')
    print(f'  {verdict} {name:12s} AUROC = {auroc:.3f}')
passed_detection = all(a >= 0.70 for a in auroc_results.values()) if auroc_results else False
print(f'\nOverall: {"✅ PASS" if passed_detection else "❌ FAIL"} (need ≥0.70 across all benchmarks)')

## 9. Mitigation — abstain mode

Re-run benchmarks with `mode='abstain', threshold=...`. Sweep threshold to find the best operating point: maximize `correct - confident_wrong`.

Definitions per question:
- **correct**: model answered correctly (and was not flagged-into-abstention)
- **confident-wrong**: model answered confidently AND incorrectly
- **abstained**: guard replaced output with uncertainty response

We want to **convert confident-wrong → abstained** while preserving **correct**.

In [ ]:
def threshold_sweep(rows, thresholds):
    """For a list of (score, correct) rows, simulate abstain mode at each threshold."""
    summary = []
    n = len(rows)
    for thr in thresholds:
        correct = 0
        confident_wrong = 0
        abstained = 0
        for r in rows:
            if r['score'] > thr:
                abstained += 1
            elif r['correct']:
                correct += 1
            else:
                confident_wrong += 1
        summary.append({
            'threshold': float(thr),
            'correct_pct':           100*correct/n,
            'confident_wrong_pct':   100*confident_wrong/n,
            'abstained_pct':         100*abstained/n,
            'trustworthiness': 100*(correct - confident_wrong)/n,
        })
    return summary

all_scores = np.concatenate([
    [r['score'] for r in rows] for rows in results.values() if rows
])
thresholds = np.percentile(all_scores, [50, 60, 70, 75, 80, 85, 90, 95])
thresholds = sorted(set(np.round(thresholds, 3).tolist()))

print('Threshold sweep — confident_wrong rate per benchmark:\n')
sweep_per_bench = {}
for bench_name, rows in results.items():
    if not rows: continue
    sw = threshold_sweep(rows, thresholds)
    sweep_per_bench[bench_name] = sw
    print(f'\n{bench_name}:')
    print(f'  {"thr":>6} {"correct%":>9} {"conf_wrong%":>12} {"abstain%":>9} {"trust":>8}')
    for row in sw:
        print(f'  {row["threshold"]:>6.2f} {row["correct_pct"]:>8.1f}% {row["confident_wrong_pct"]:>11.1f}% '
              f'{row["abstained_pct"]:>8.1f}% {row["trustworthiness"]:>+7.1f}')

# Find best threshold across non-MMLU benchmarks (MMLU is capability control, not hallucination)
halluc_benches = ['truthfulqa', 'haluval', 'simpleqa']
halluc_rows = sum((results[b] for b in halluc_benches if results[b]), start=[])
if halluc_rows:
    sw_all = threshold_sweep(halluc_rows, thresholds)
    best = max(sw_all, key=lambda r: r['trustworthiness'])
    print(f'\n🎯 Best threshold (hallucination benchmarks combined): {best["threshold"]:.2f}')
    print(f'   correct={best["correct_pct"]:.1f}% conf_wrong={best["confident_wrong_pct"]:.1f}% '
          f'abstain={best["abstained_pct"]:.1f}% trustworthiness={best["trustworthiness"]:+.1f}')
    CFG['guard_threshold'] = best['threshold']

## 10. Mitigation impact — confident-wrong rate before vs after

At the best threshold, what's the impact?

In [ ]:
thr = CFG['guard_threshold']
print(f'Using threshold = {thr:.2f}\n')

headline_table = {}
for bench_name, rows in results.items():
    if not rows: continue
    n = len(rows)
    # Baseline
    base_correct  = sum(1 for r in rows if r['correct'])
    base_wrong    = n - base_correct
    # With abstain
    correct = sum(1 for r in rows if r['score'] <= thr and r['correct'])
    wrong = sum(1 for r in rows if r['score'] <= thr and not r['correct'])
    abstain = sum(1 for r in rows if r['score'] > thr)
    headline_table[bench_name] = {
        'n': n,
        'baseline_correct_pct':       100 * base_correct / n,
        'baseline_confwrong_pct':     100 * base_wrong / n,
        'guard_correct_pct':          100 * correct / n,
        'guard_confwrong_pct':        100 * wrong / n,
        'guard_abstain_pct':          100 * abstain / n,
        'confwrong_reduction_pct':    100 * (base_wrong - wrong) / max(1, base_wrong),
    }

print(f'{"benchmark":>14} {"correct":>9} {"conf_wrong base":>16} {"conf_wrong guard":>17} {"abstained":>10} {"reduction":>10}')
print('-' * 88)
for name, t in headline_table.items():
    print(f'{name:>14} '
          f'{t["baseline_correct_pct"]:>8.1f}% '
          f'{t["baseline_confwrong_pct"]:>15.1f}% '
          f'{t["guard_confwrong_pct"]:>16.1f}% '
          f'{t["guard_abstain_pct"]:>9.1f}% '
          f'{t["confwrong_reduction_pct"]:>+9.1f}%')

# Pass criterion: confident_wrong reduction ≥30% on hallucination benchmarks
halluc_reductions = [headline_table[b]['confwrong_reduction_pct']
                     for b in halluc_benches if b in headline_table]
passed_mitigation = (len(halluc_reductions) > 0
                     and np.mean(halluc_reductions) >= 30.0)
print(f'\nQuestion 2: Mitigation works?')
print(f'  Mean confident-wrong reduction (halluc benches): {np.mean(halluc_reductions):.1f}%')
print(f'  Verdict: {"✅ PASS" if passed_mitigation else "❌ FAIL"} (need ≥30% reduction)')

## 11. Capability regression check (MMLU)

Does abstaining on MMLU questions hurt the score? MMLU is knowledge — guard should NOT trigger heavily on it. If MMLU score drops > 3pp, the threshold is too aggressive.

In [ ]:
if 'mmlu' in headline_table:
    t = headline_table['mmlu']
    base = t['baseline_correct_pct']
    guard = t['guard_correct_pct']                    # only counting correct AND not-abstained
    abst = t['guard_abstain_pct']
    # "Effective MMLU score" — counting abstentions as wrong (worst case)
    effective = guard
    delta = effective - base
    print(f'MMLU baseline (no guard):  {base:.1f}%')
    print(f'MMLU with abstain @ thr:   {effective:.1f}%  (abstained on {abst:.1f}% of questions)')
    print(f'Δ MMLU:                    {delta:+.1f}pp')
    passed_capability = delta >= -3.0
    print(f'\nVerdict: {"✅ PASS" if passed_capability else "❌ FAIL"} (need Δ ≥ -3pp)')
else:
    passed_capability = False
    print('MMLU not loaded — cannot evaluate capability regression.')

## 12. Latency benchmark — competitive cost?

How long does `guard.score()` take per prompt? For competitive parity with Galileo Luna-2 ($0.02/1M tokens, 152ms median latency), we need < 50ms per score (since we're in-process and don't add a separate LLM-judge forward).

In [ ]:
# Warmup
for _ in range(3):
    guard.score('What is the capital of France?')

torch.cuda.synchronize()
n_trials = 20
test_prompt = 'What can you tell me about a topic?'
t0 = time.time()
for _ in range(n_trials):
    _ = guard.score(test_prompt)
torch.cuda.synchronize()
t = time.time() - t0
score_latency_ms = 1000 * t / n_trials

# baseline generation latency (reference)
t0 = time.time()
_ = model_generate(test_prompt, max_new_tokens=64)
torch.cuda.synchronize()
gen_latency_ms = 1000 * (time.time() - t0)

overhead_pct = 100 * score_latency_ms / max(gen_latency_ms, 1)
print(f'Score latency:        {score_latency_ms:.1f} ms / call')
print(f'Generation latency:   {gen_latency_ms:.0f} ms / 64 tokens')
print(f'Score overhead:       {overhead_pct:.1f}% of generation cost')
passed_latency = score_latency_ms < 50.0
print(f'\nQuestion 3: Competitive?')
print(f'  Verdict: {"✅ PASS" if passed_latency else "❌ FAIL"} (need < 50 ms/score)')

## 13. Headline figure — the marketing artifact

Single chart that goes on landing page + tweet.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))

names = []
base_wrong = []
guard_wrong = []
guard_abst = []
for n in halluc_benches:
    if n not in headline_table: continue
    t = headline_table[n]
    names.append(n)
    base_wrong.append(t['baseline_confwrong_pct'])
    guard_wrong.append(t['guard_confwrong_pct'])
    guard_abst.append(t['guard_abstain_pct'])

x = np.arange(len(names))
w = 0.4

ax.bar(x - w/2, base_wrong, w, color='#d62728', alpha=0.85,
       label='Baseline: confidently WRONG')
ax.bar(x + w/2, guard_wrong, w, color='#ff7f0e', alpha=0.85,
       label=f'+ HallucinationGuard (abstain @ thr={thr:.2f}): confidently WRONG')
ax.bar(x + w/2, guard_abst, w, bottom=guard_wrong,
       color='#2ca02c', alpha=0.55,
       label='+ HallucinationGuard: ABSTAINED honestly')

for i, (b, g, a) in enumerate(zip(base_wrong, guard_wrong, guard_abst)):
    ax.text(i - w/2, b + 1, f'{b:.0f}%', ha='center', fontsize=10, color='#a01010')
    ax.text(i + w/2, g + a + 1, f'{g:.0f}%+{a:.0f}%', ha='center', fontsize=10, color='#604010')
    reduction = 100*(b-g)/max(1,b)
    ax.text(i + w/2, -3, f'-{reduction:.0f}% wrong',
             ha='center', fontsize=10, color='#0a5a0a', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=11)
ax.set_ylabel('% of questions', fontsize=11)
ax.set_title('Qwen3.6-27B with HallucinationGuard — confident-wrong rate plummets, '
             'replaced by honest abstention', fontsize=13)
ax.legend(loc='upper right', fontsize=10)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(-7, max([b+5 for b in base_wrong] + [g+a+5 for g,a in zip(guard_wrong, guard_abst)]) + 5)

plt.tight_layout()
fig.savefig(LOCAL_OUT / 'headline.png', dpi=200, bbox_inches='tight')
fig.savefig(LOCAL_OUT / 'headline.pdf', bbox_inches='tight')
plt.show()

## 14. Final verdict + save artifacts

If all 3 questions pass → green light 90-day product roadmap. If 2/3 → yellow (tunable). If 1/3 or 0 → red (pivot).

In [ ]:
verdict = {
    'detection_works':   passed_detection,
    'mitigation_works':  passed_mitigation,
    'capability_intact': passed_capability,
    'competitive':       passed_latency,
    'auroc_per_benchmark':  auroc_results,
    'headline_table':       headline_table,
    'best_threshold':       float(thr),
    'score_latency_ms':     float(score_latency_ms),
    'gen_latency_ms':       float(gen_latency_ms),
    'cfg':                  CFG,
}
n_passed = sum([passed_detection, passed_mitigation, passed_capability, passed_latency])
verdict['summary'] = f'{n_passed}/4 thresholds passed'
if n_passed == 4:
    verdict['decision'] = '🟢 GREEN LIGHT — proceed with 90-day product roadmap'
elif n_passed == 3:
    verdict['decision'] = '🟡 TUNABLE — fix the failing dimension before launch'
else:
    verdict['decision'] = '🔴 PIVOT — current SAE feature does not generalize, narrow scope or rebuild'

(LOCAL_OUT / 'verdict.json').write_text(json.dumps(verdict, indent=2, default=str))
(LOCAL_OUT / 'results_full.json').write_text(json.dumps({
    name: [{k: (v.tolist() if isinstance(v, np.ndarray) else v)
             for k, v in r.items()} for r in rows]
    for name, rows in results.items() if rows
}, indent=2, default=str))

print('=' * 70)
print(f'  HallucinationGuard PoC verdict — {verdict["summary"]}')
print('=' * 70)
print(f'  Detection AUROC   : {"✅" if passed_detection else "❌"}  '
      f'({"all" if passed_detection else "some"} ≥ 0.70)')
print(f'  Mitigation impact : {"✅" if passed_mitigation else "❌"}  '
      f'({np.mean(halluc_reductions):.1f}% confwrong reduction)')
print(f'  Capability intact : {"✅" if passed_capability else "❌"}  '
      f'(Δ MMLU = {headline_table.get("mmlu",{}).get("guard_correct_pct",0) - headline_table.get("mmlu",{}).get("baseline_correct_pct",0):+.1f}pp)')
print(f'  Competitive cost  : {"✅" if passed_latency else "❌"}  '
      f'({score_latency_ms:.1f}ms/score, target < 50ms)')
print('=' * 70)
print(f'  {verdict["decision"]}')
print('=' * 70)

# Push to HF
api = HfApi()
create_repo(CFG['hf_results_repo'], exist_ok=True, private=False, token=HF_TOKEN, repo_type='dataset')
api.upload_folder(
    folder_path=str(LOCAL_OUT),
    repo_id=CFG['hf_results_repo'],
    repo_type='dataset',
    token=HF_TOKEN,
)
print(f'\n✅ Results pushed to https://huggingface.co/datasets/{CFG["hf_results_repo"]}')

## 15. Where this leaves us

**If 🟢 GREEN LIGHT (4/4 passed)**:
1. The headline figure goes on `openinterp.org/products/hallucinationguard`
2. Notebook becomes the public proof-of-concept (linked from README)
3. Begin 90-day product roadmap (S1: SDK → S2: probes → S3: launch)
4. Submit ICML MI Workshop paper using these numbers

**If 🟡 TUNABLE (3/4)**:
- If detection passed but mitigation didn't: tune threshold, try ablation strength sweeps, train a small linear probe on top of the SAE feature for better calibration
- If mitigation passed but capability dropped: tighter threshold, per-domain thresholds (general questions vs specialized)
- If competitive failed (latency): batch the score call, fp16 the SAE encoder, or train a smaller distilled probe

**If 🔴 PIVOT (≤2/4)**:
- Detection failed → Ferrando entity probe doesn't generalize beyond entity questions. Product is *EntityRecognitionGuard*, not general HallucinationGuard. Narrower TAM but still real.
- Try linear probe (LR ceiling 0.887 in notebook 28) instead of SAE feature — gives stronger signal at cost of less interpretability
- Pivot toward RAG-grounded scoring (use SAE feature alongside retrieved-context match)

## Reading & next steps

- Notebook 28 (full Ferrando baselines): `OpenInterpretability/notebooks/28_paper_baselines_qwen36_27b.ipynb`
- Pearson_CE methodology (paper-1): `notebooks/17b_crosscoder_model_diff_papergrade.ipynb`
- Source SAE: <https://huggingface.co/caiovicentino1/qwen36-27b-sae-papergrade>

Once verdict is in, see `MEMORY.md` entry on this run for the strategic decision.